In [1]:
from pyspark.sql import SparkSession
#import os,tempfile,shutil

spark=(SparkSession.builder
    .appName("Tranformation")
    .master("local[*]")
    .getOrCreate())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 15:34:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/27 15:34:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
from pyspark.sql import functions as F 
df_emp=(spark.read
        .format("csv")
        .option("header","true")
        .option("sep","|")
        .option("inferSchema","true")
        .option("nullvalue","NULL")
        .load("employees.csv"))

df_dept=(spark.read
        .format("csv")
        .option("header","true")
        .option("sep","|")
        .option("inferSchema","true")
        .option("nullvalue","NULL")
        .load("departments.csv"))

In [5]:
df_emp.show(5,truncate=False)
df_dept.show(5,truncate=False)

+------+----------+---------+------------------------+-------------+--------------------+--------+-----+---------+----------+-------------------+---------+------------------+---------------+
|emp_id|first_name|last_name|email                   |department_id|job_title           |salary  |bonus|is_active|hire_date |last_login         |city     |performance_rating|employment_type|
+------+----------+---------+------------------------+-------------+--------------------+--------+-----+---------+----------+-------------------+---------+------------------+---------------+
|1001  |Anuj      |Shahdeo  |anuj.shahdeo@company.com|10           |Senior Data Engineer|125000.5|10000|true     |2019-04-15|2024-02-01 10:30:00|Mumbai   |4.7               |Full-Time      |
|1002  |Riya      |Verma    |riya.verma@company.com  |20           |HR Manager          |95000.0 |7000 |true     |2018-07-10|2024-02-02 09:15:00|Delhi    |4.3               |Full-Time      |
|1003  |John      |Doe      |john.doe@company

In [6]:
# Schema 
df_dept.printSchema()

root
 |-- department_id: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- manager_name: string (nullable = true)
 |-- budget: integer (nullable = true)
 |-- established_year: integer (nullable = true)



In [7]:
df_emp.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- job_title: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- bonus: integer (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- last_login: timestamp (nullable = true)
 |-- city: string (nullable = true)
 |-- performance_rating: double (nullable = true)
 |-- employment_type: string (nullable = true)



# Transformation 

1. select() - Pick only a few columns ( this is column pruning concept concept)
    - Pick specific column 
    - create expression 
    - Rename column (via alias)
    - Transform column s

    # It is a narrow transformation 
        - No shuffle
        - Operate row by row 

    df.select(*cols)

2. withColumn()

    withColumn used to :
        - add new column 
        - Modify an existing column 
        - Apply transformation to column 
    # Return new data frame 

    # df.withColumn("new column_name", column_expression)
        - new column_name -> Name of new column 
        - column expression -> Must be a Column Object 

    

In [9]:
df1=df_emp.select("emp_id","first_name","department_id","salary")
df1.show(5,truncate=False)

+------+----------+-------------+--------+
|emp_id|first_name|department_id|salary  |
+------+----------+-------------+--------+
|1001  |Anuj      |10           |125000.5|
|1002  |Riya      |20           |95000.0 |
|1003  |John      |10           |105000.0|
|1004  |Sara      |30           |88000.0 |
|1005  |Amit      |40           |72000.0 |
+------+----------+-------------+--------+
only showing top 5 rows



In [10]:
df1.explain("formatted")

== Physical Plan ==
Scan csv  (1)


(1) Scan csv 
Output [4]: [emp_id#195, first_name#196, department_id#199, salary#201]
Batched: false
Location: InMemoryFileIndex [file:/Users/anujshahdeo/Documents/GithubRepos/Spark_DE/Module_4/employees.csv]
ReadSchema: struct<emp_id:int,first_name:string,department_id:int,salary:double>




In [31]:
from pyspark.sql.functions import col,lit


In [25]:
# Example 1 

# Add New column -> Add 10 % salary increment 

df2=df_emp.withColumn("Salary_after_increment",col("salary")*1.10)


In [27]:
df2.select("emp_id","salary","Salary_after_increment").show(5,truncate=False)

+------+--------+----------------------+
|emp_id|salary  |Salary_after_increment|
+------+--------+----------------------+
|1001  |125000.5|137500.55000000002    |
|1002  |95000.0 |104500.00000000001    |
|1003  |105000.0|115500.00000000001    |
|1004  |88000.0 |96800.00000000001     |
|1005  |72000.0 |79200.0               |
+------+--------+----------------------+
only showing top 5 rows



In [29]:
df2.explain("formatted")

== Physical Plan ==
* Project (2)
+- Scan csv  (1)


(1) Scan csv 
Output [14]: [emp_id#195, first_name#196, last_name#197, email#198, department_id#199, job_title#200, salary#201, bonus#202, is_active#203, hire_date#204, last_login#205, city#206, performance_rating#207, employment_type#208]
Batched: false
Location: InMemoryFileIndex [file:/Users/anujshahdeo/Documents/GithubRepos/Spark_DE/Module_4/employees.csv]
ReadSchema: struct<emp_id:int,first_name:string,last_name:string,email:string,department_id:int,job_title:string,salary:double,bonus:int,is_active:boolean,hire_date:date,last_login:timestamp,city:string,performance_rating:double,employment_type:string>

(2) Project [codegen id : 1]
Output [15]: [emp_id#195, first_name#196, last_name#197, email#198, department_id#199, job_title#200, salary#201, bonus#202, is_active#203, hire_date#204, last_login#205, city#206, performance_rating#207, employment_type#208, (salary#201 * 1.1) AS Salary_after_increment#538]
Input [14]: [emp_id#195, 

In [30]:
# Example 2 Modify existing column 
df3=df_emp.withColumn(
    "salary",col("salary")*1.10
)
df3.select("emp_id","salary").show(5,truncate=False)

+------+------------------+
|emp_id|salary            |
+------+------------------+
|1001  |137500.55000000002|
|1002  |104500.00000000001|
|1003  |115500.00000000001|
|1004  |96800.00000000001 |
|1005  |79200.0           |
+------+------------------+
only showing top 5 rows



In [32]:
df4=df_emp.withColumn(
    "new_bonus",lit(10)
)

In [33]:
df4.show()

+------+----------+---------+--------------------+-------------+--------------------+--------+-----+---------+----------+-------------------+---------+------------------+---------------+---------+
|emp_id|first_name|last_name|               email|department_id|           job_title|  salary|bonus|is_active| hire_date|         last_login|     city|performance_rating|employment_type|new_bonus|
+------+----------+---------+--------------------+-------------+--------------------+--------+-----+---------+----------+-------------------+---------+------------------+---------------+---------+
|  1001|      Anuj|  Shahdeo|anuj.shahdeo@comp...|           10|Senior Data Engineer|125000.5|10000|     true|2019-04-15|2024-02-01 10:30:00|   Mumbai|               4.7|      Full-Time|       10|
|  1002|      Riya|    Verma|riya.verma@compan...|           20|          HR Manager| 95000.0| 7000|     true|2018-07-10|2024-02-02 09:15:00|    Delhi|               4.3|      Full-Time|       10|
|  1003|      J

In [34]:
from pyspark.sql.functions import coalesce

In [37]:
df5=df_emp.withColumn(
    "bonus_clean",
    coalesce(col("bonus"),lit(0))
)

df5.select("emp_id","salary","bonus","bonus_clean").show()

+------+--------+-----+-----------+
|emp_id|  salary|bonus|bonus_clean|
+------+--------+-----+-----------+
|  1001|125000.5|10000|      10000|
|  1002| 95000.0| 7000|       7000|
|  1003|105000.0| 5000|       5000|
|  1004| 88000.0| 4000|       4000|
|  1005| 72000.0| 3000|       3000|
|  1006| 65000.0| NULL|          0|
|  1007|135000.0|15000|      15000|
|  1008|110000.0| 6000|       6000|
|  1009| 99000.0| 8000|       8000|
|  1010| 87000.0| 3500|       3500|
|  1011|150000.0|20000|      20000|
|  1012| 92000.0| 4500|       4500|
|  1013| 83000.0| 5000|       5000|
|  1014|130000.0|12000|      12000|
|  1015|115000.0| 9000|       9000|
+------+--------+-----+-----------+



In [38]:
from pyspark.sql.functions import when
df6=df_emp.withColumn(
    "salary_band",
    when(col("salary")>120000,"High")
    .when(col("salary")>80000,"medium")
    .otherwise("low")
)

df6.select("emp_id","salary","salary_band").show()

+------+--------+-----------+
|emp_id|  salary|salary_band|
+------+--------+-----------+
|  1001|125000.5|       High|
|  1002| 95000.0|     medium|
|  1003|105000.0|     medium|
|  1004| 88000.0|     medium|
|  1005| 72000.0|        low|
|  1006| 65000.0|        low|
|  1007|135000.0|       High|
|  1008|110000.0|     medium|
|  1009| 99000.0|     medium|
|  1010| 87000.0|     medium|
|  1011|150000.0|       High|
|  1012| 92000.0|     medium|
|  1013| 83000.0|     medium|
|  1014|130000.0|       High|
|  1015|115000.0|     medium|
+------+--------+-----------+



In [41]:
df_emp.select("*").show()

+------+----------+---------+--------------------+-------------+--------------------+--------+-----+---------+----------+-------------------+---------+------------------+---------------+
|emp_id|first_name|last_name|               email|department_id|           job_title|  salary|bonus|is_active| hire_date|         last_login|     city|performance_rating|employment_type|
+------+----------+---------+--------------------+-------------+--------------------+--------+-----+---------+----------+-------------------+---------+------------------+---------------+
|  1001|      Anuj|  Shahdeo|anuj.shahdeo@comp...|           10|Senior Data Engineer|125000.5|10000|     true|2019-04-15|2024-02-01 10:30:00|   Mumbai|               4.7|      Full-Time|
|  1002|      Riya|    Verma|riya.verma@compan...|           20|          HR Manager| 95000.0| 7000|     true|2018-07-10|2024-02-02 09:15:00|    Delhi|               4.3|      Full-Time|
|  1003|      John|      Doe|john.doe@company.com|           10| 